# Server Metrics EDA — Anomaly Detection Pipeline
Loads generated data, visualizes metrics over time, and inspects injected anomalies.

In [ ]:
import yaml
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.generator import generate_metrics
from src.data.validator import validate_metrics
from src.features.builder import build_features

sns.set_style("darkgrid")
%matplotlib inline

In [ ]:
with open("../configs/config.yaml") as f:
    config = yaml.safe_load(f)

df = generate_metrics(config)
validation = validate_metrics(df)
print(f"Valid: {validation['valid']} | Rows: {validation['n_rows']} | Cols: {validation['n_columns']}")
df.head()

In [ ]:
anomaly_pct = df["is_anomaly"].mean() * 100
n_anomalies = df["is_anomaly"].sum()
print(f"Total points: {len(df):,} | Anomalies: {n_anomalies} ({anomaly_pct:.1f}%)")

## Time Series — All Metrics

In [ ]:
metrics = ["cpu_pct", "mem_pct", "disk_io", "latency_ms", "error_rate"]
fig, axes = plt.subplots(len(metrics), 1, figsize=(14, 12), sharex=True)

for ax, col in zip(axes, metrics):
    ax.plot(df["timestamp"], df[col], alpha=0.7, linewidth=0.5, label=col)
    anomaly_mask = df["is_anomaly"] == 1
    ax.scatter(
        df.loc[anomaly_mask, "timestamp"],
        df.loc[anomaly_mask, col],
        color="red", s=4, alpha=0.6, label="anomaly"
    )
    ax.set_ylabel(col)
    ax.legend(loc="upper right", fontsize=7)

axes[-1].set_xlabel("Timestamp")
plt.tight_layout()
plt.show()

## Distribution of Each Metric

In [ ]:
fig, axes = plt.subplots(1, len(metrics), figsize=(16, 4))
for ax, col in zip(axes, metrics):
    normal = df.loc[df["is_anomaly"] == 0, col]
    anomalous = df.loc[df["is_anomaly"] == 1, col]
    ax.hist(normal, bins=40, alpha=0.6, label="normal", density=True)
    ax.hist(anomalous, bins=20, alpha=0.6, label="anomaly", density=True, color="red")
    ax.set_title(col)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## Zoomed View — Last 2 Days

In [ ]:
last_2days = df.set_index("timestamp").last("2D")
fig, axes = plt.subplots(len(metrics), 1, figsize=(14, 12), sharex=True)

for ax, col in zip(axes, metrics):
    ax.plot(last_2days.index, last_2days[col], alpha=0.7, linewidth=0.6)
    anomaly_mask = last_2days["is_anomaly"] == 1
    ax.scatter(
        last_2days.index[anomaly_mask],
        last_2days.loc[anomaly_mask, col],
        color="red", s=8, alpha=0.8
    )
    ax.set_ylabel(col)

axes[-1].set_xlabel("Timestamp")
plt.suptitle("Last 2 Days — Zoomed")
plt.tight_layout()
plt.show()

## Feature Engineering — Output Shape

In [ ]:
feature_df = build_features(df, config)
print(f"Features shape: {feature_df.shape}")
print(f"Feature columns ({len(feature_df.columns)}):")
for c in feature_df.columns[:10]:
    print(f"  {c}")
print(f"  ... (+ {len(feature_df.columns) - 10} more)")

## Correlation Heatmap (Top Features)

In [ ]:
corr = feature_df.corr()[feature_df.columns[:12]].iloc[:12]
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Feature Correlation Matrix (first 12 columns)")
plt.tight_layout()
plt.show()